In [1]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("Pyspark Exercises").getOrCreate()

In [29]:
from google.colab import files

uploaded =files.upload()

Saving appointments.csv to appointments.csv


In [91]:
patients_df=spark.read.csv("patients.csv",header=True,inferSchema=True)

appointments_df=spark.read.csv("appointments.csv",header=True,inferSchema=True)

preference_df=spark.read.option("multiLine","true").json("patient_preferences.json")

In [92]:
patients_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)



In [93]:
appointments_df.printSchema()

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: timestamp (nullable = true)
 |-- consult: integer (nullable = true)
 |-- status: string (nullable = true)



In [94]:
patients_df.show()
appointments_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [95]:
preference_df.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [96]:
print("Patients Count :",patients_df.count())
print("Appointments Count :",appointments_df.count())

Patients Count : 10
Appointments Count : 10


In [ ]:
patients_df.show(5)

In [ ]:
patients_df.select("city").distinct().show()

In [ ]:
appointments_df.select("department").distinct().show()

In [ ]:
patients_df.write.mode("overwrite").parquet("patients_parquet")

In [ ]:
patients_parquet = spark.read.parquet("patients_parquet")

patients_parquet.show()

In [ ]:
print("CSV count:",patients_df.count())
print("parquet count:",patients_parquet.count())

In [ ]:
patients_df.filter(patients_df.city=="Hyderabad").show()

In [ ]:
patients_df.filter(patients_df.gender=="Female").show()

In [ ]:
patients_df.filter(patients_df.age>40).show()

In [ ]:
appointments_df.filter(appointments_df.status=="Completed").show()

In [ ]:
appointments_df.filter(appointments_df.status=="Pending").show()

In [ ]:
appointments_df.filter(appointments_df.consult>1500).show()

In [ ]:
patients_df.filter(patients_df.insurance_status=="Active").show()

In [ ]:
patients_df.filter(patients_df.insurance_status=="Inactive").show()

In [ ]:
patients_df.filter(patients_df.blood_group=="O+").show()

In [ ]:
appointments_df.filter(appointments_df.department=="Cardiology").show()

In [ ]:
patients_df.filter(patients_df.city.isNull()).show()

In [ ]:
patients_df.filter(patients_df.blood_group.isNull()).show()

In [ ]:
appointments_df.filter(appointments_df.consult.isNull()).show()

In [ ]:
from pyspark.sql.functions import *

patients_df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in patients_df.columns
]).show()

In [ ]:
patients_df =patients_df.na.fill({
    "city":"Unknown",
    "blood_group":"Not Available",

})

patients_df.show()

In [ ]:
appointments_df.na.fill({
    "consult":0
}).show()

In [ ]:
appointments_df.na.drop(subset=["consult"]).show()

In [ ]:
patients_df = patients_df.withColumn(
    "data_quality_status",
    when(
        (col("city")=="Unknown") |
        (col("blood_group")=="Not Available"),
        "Incomplete"
    ).otherwise("Complete")
)

patients_df.show()

In [ ]:
patients_df.groupBy(
    "data_quality_status"
).count().show()

In [ ]:
patients_df.select(upper("patient_name")).show()

In [ ]:
patients_df.select(lower("patient_name")).show()

In [ ]:
patients_df.select("patient_name",length("patient_name")).show()

In [ ]:
patients_df.select(
    substring("patient_name",1,3).alias("Substring")
).show()

In [ ]:
patients_df=patients_df.withColumn(
    "insurance_flag",when(
        col("insurance_status")=="Active",1
    ).otherwise(0)
)

patients_df.show()

In [ ]:
patients_df=patients_df.withColumn(
    "senior_citizen",when(
        col("age_group")=="Senior","yes"
    ).otherwise("no")
)

patients_df.show()

In [ ]:
patients_df.select(concat_ws(
    " - ","patient_name","city"
).alias("Concat")).show(truncate=False)

In [ ]:
patients_df.select(
    "patient_name",
    trim(patients_df.patient_name).alias("trim")
).show()

In [ ]:
patients_df.select(upper("city")).show()

In [ ]:
patients_df.groupBy("city").count().show()

In [ ]:
patients_df.groupBy("gender").count().show()

In [ ]:
patients_df.groupBy("blood_group").count().show()

In [ ]:
appointments_df.groupBy("department").count().show()

In [ ]:
patients_df.groupBy("city").agg(avg("age")).show()

In [ ]:
patients_df.groupBy("city").agg(max("age")).show()

In [ ]:
patients_df.groupBy("city").agg(min("age")).show()

In [ ]:
appointments_df.groupBy("department").agg(avg("consult")).show()

In [ ]:
appointments_df.groupBy("department").agg(sum("consult")).show()

In [ ]:
appointments_df.groupBy("department").agg(sum
  ("consult").alias("revenue")).orderBy(
     desc("revenue")).show(1)

In [ ]:
patients_df.join(
    appointments_df,"patient_id","inner"
).show()

In [ ]:
patients_df.join(
    appointments_df,"patient_id","left"
).show()

In [ ]:
patients_df.join(
    appointments_df,"patient_id","right"
).show()

In [ ]:
patients_df.join(
    appointments_df,"patient_id","full"
).show()

In [ ]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left_anti"
).show()

In [ ]:
appointments_df.join(
    patients_df,"patient_id","left_anti"
).show()

In [ ]:
appointments_df.groupBy(
    "patient_id"
).count().show()

In [ ]:
appointments_df.groupBy(
    "patient_id"
).agg(sum("consult")).show()

In [ ]:
appointments_df.groupBy(
    "patient_id"
).agg(
    sum("consult").alias("total_fee")
).orderBy(
    desc("total_fee")
).show(1)

In [ ]:
from pyspark.sql.window import Window

window_spec=Window.orderBy(
    col("consult").desc()
)

appointments_df.withColumn(
    "rank",rank().over(window_spec)
).show()

In [ ]:
appointments_df.withColumn(
    "dense_rank",dense_rank().over(window_spec)
).show()

In [ ]:
appointments_df.withColumn(
    "row_number",row_number().over(window_spec)
).show()

In [ ]:
patient_spending = appointments_df.groupBy("patient_id") \
    .agg(sum("consult").alias("total_fee"))

patient_spending.orderBy(
    desc("total_fee")
).show(1)

In [ ]:
patient_spending.orderBy(
    desc("total_fee")
).show(3)

In [ ]:
patient_city = patients_df.join(
    patient_spending,
    "patient_id"
)

window_city = Window.partitionBy(
    "city"
).orderBy(desc("total_fee"))

patient_city.withColumn(
    "rank",
    rank().over(window_city)
).filter(
    col("rank")==1
).show()

In [ ]:
window_city = Window.partitionBy(
    "city"
).orderBy("total_fee")

patient_city.withColumn(
    "rank",
    rank().over(window_city)
).filter(
    col("rank")==1
).show()

In [ ]:
window_run = Window.orderBy(
    "appointment_date"
)

appointments_df.withColumn(
    "running_total",
    sum("consult").over(window_run)
).show()

In [ ]:
appointments_df.withColumn(
    "next_fee",
    lead("consult").over(window_run)
).show()

In [ ]:
appointments_df.withColumn(
    "previous_fee",
    lag("consult").over(window_run)
).show()

In [252]:
preference_df.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [251]:
preference_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [ ]:
flat_df=preference_df.select(
    "patient_id","preferred_hospital",
    col("contact.email").alias("email"),
    col("contact.phone").alias("phone")
)

flat_df.show()

In [ ]:
flat_df.filter(col("email").isNull()).show()

In [ ]:
flat_df.filter(col("phone").isNull()).show()

In [ ]:
flat_df=flat_df.na.fill({
    "phone":"Not Provided",
    "email":"Not Provided"
}
)

flat_df.show()

In [ ]:
temp=patients_df.join(
    flat_df,"patient_id"
)

temp.show()

In [ ]:
patients_df.createOrReplaceTempView("patients")

In [ ]:
appointments_df.createOrReplaceTempView("appointments")

In [ ]:
spark.sql(
    """select * from patients"""
).show()

In [ ]:
spark.sql("""
  select * from patients
  where city="Hyderabad"
""").show()

In [ ]:
spark.sql("""
  select city,count(*) from patients
  group by city;
""").show()

In [ ]:
spark.sql("""
  select department,count(*) from appointments
  group by department;
""").show()

In [ ]:
spark.sql("""
  select department,avg(consult) from appointments
  group by department;
""").show()

In [ ]:
spark.sql("""
  select max(consult) from appointments
""").show()

In [ ]:
spark.sql("""
  select patient_id,count(*) from appointments
  group by patient_id;
""").show()

In [ ]:
spark.sql("""
  select patient_id,consult from appointments
  order by consult desc;
""").show(5)

In [ ]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

preferences_df = spark.read.option(
    "multiline",
    "true"
).json("patient_preferences.json")

In [ ]:
patients_df = patients_df.fillna({
    "city":"Unknown",
    "blood_group":"Not Available"
})

appointments_df = appointments_df.fillna({
    "consult":0
})

In [ ]:
final_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    flat_df,
    "patient_id",
    "left"
)

final_df.show()

In [ ]:
patients_df=patients_df.withColumn(
    "age_group",when(col("age")<30,"Young")
    .when(col("age")<50,"Adult")
    .otherwise("Senior")
)

patients_df.show()

In [ ]:
final_df.withColumn(
    "revenue",col("consult")
).show()

In [ ]:
patient_report =final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("consult").alias("total_spending")
)

patient_report.show()

In [ ]:
department_report =final_df.groupBy(
    "department"
).agg(
    sum("consult").alias("total_revenue")
)

department_report.show()

In [237]:
final_df.write.mode("overwrite").parquet(
    "hospital_analytics_output"
)

In [250]:
print("--- HOSPITAL ANALYTICS REPORT ---")

print("Total Patients:")
print(patients_df.count(),end="\n\n")

print("Total Appointments:")
print(appointments_df.count(),end="\n\n")

print("Total Revenue:")
final_df.agg(
    sum("consult")
).show()

print("Top Spending Patients:")
patient_report.orderBy(
    desc("total_spending")
).show(5)

print("\nDepartment Revenue:")
department_report.orderBy(
    desc("total_revenue")
).show()

--- HOSPITAL ANALYTICS REPORT ---
Total Patients:
10

Total Appointments:
10

Total Revenue:
+------------+
|sum(consult)|
+------------+
|       14000|
+------------+

Top Spending Patients:
+----------+------------+--------------+
|patient_id|patient_name|total_spending|
+----------+------------+--------------+
|       110| Nisha Reddy|          2500|
|       101|Rahul Sharma|          2500|
|       104| Sneha Patel|          2500|
|       107| Arjun Verma|          2000|
|       102| Priya Reddy|          2000|
+----------+------------+--------------+
only showing top 5 rows

Department Revenue:
+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
|  Neurology|         4000|
| Cardiology|         3000|
|Dermatology|         2000|
|       NULL|         NULL|
+-----------+-------------+

